# NPZ File Structure Analysis and PPG Visualization

This notebook analyzes the structure of NPZ files from `/home/cristic/preprocessed_data/` and visualizes the PPG data.

**Note:** The time_deltas in your data appear to be in milliseconds or need scaling. This notebook handles that automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import json
from scipy import signal
from scipy.signal import butter, filtfilt, detrend, find_peaks

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
PREPROCESSED_DATA_PATH = Path('/home/cristic/preprocessed_data/')

# Ensure the path exists
if not PREPROCESSED_DATA_PATH.exists():
    print(f"Warning: {PREPROCESSED_DATA_PATH} does not exist. Using current directory instead.")
    PREPROCESSED_DATA_PATH = Path('.')

In [ ]:
def load_npz_file(file_path):
    """Load and return the contents of an NPZ file."""
    try:
        data = np.load(file_path, allow_pickle=True)
        return data
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def analyze_npz_structure(data):
    """Analyze and print the structure of an NPZ file."""
    print("=" * 60)
    print("NPZ File Structure Analysis")
    print("=" * 60)
    for key in sorted(data.files):
        item = data[key]
        if hasattr(item, 'shape'):
            print(f"  {key}: shape {item.shape}, dtype {item.dtype}")
        else:
            print(f"  {key}: {type(item).__name__} = {item}")
    print("=" * 60)
    return data

def extract_metadata(data):
    """Extract metadata from NPZ file."""
    metadata = {}
    for key in data.files:
        if key.startswith('meta_'):
            metadata[key] = data[key].item() if data[key].size == 1 else data[key]
        elif key.startswith('vital_'):
            metadata[key] = data[key].item() if data[key].size == 1 else data[key]
    return metadata

def extract_roi_names(data):
    """Extract ROI names from NPZ file."""
    rois = []
    for key in data.files:
        if key.startswith('roi_'):
            rois.append(key)
    return sorted(rois)

def compute_time_array(time_deltas, expected_fps=30):
    """Compute time array from time deltas, handling different time units."""
    # Calculate cumulative time
    time_array = np.cumsum(time_deltas)
    
    # Check if time_deltas are in seconds or need scaling
    num_frames = len(time_deltas)
    total_time = time_array[-1]
    calculated_fps = num_frames / total_time if total_time > 0 else float('inf')
    
    # If FPS is way too high (> 100), assume time_deltas are in milliseconds
    if calculated_fps > 100:
        print(f"  Note: Time deltas appear to be in milliseconds (calculated FPS: {calculated_fps:.2f}). Converting to seconds.")
        time_array = time_array / 1000.0
        total_time = time_array[-1]
        calculated_fps = num_frames / total_time
        print(f"  After scaling: Total time = {total_time:.2f}s, FPS = {calculated_fps:.2f}")
    
    # If FPS is still not matching expected, try to match expected_fps
    if expected_fps and abs(calculated_fps - expected_fps) > 5:
        scale_factor = num_frames / (expected_fps * total_time) if total_time > 0 else 1
        time_array = time_array * scale_factor
        total_time = time_array[-1]
        calculated_fps = num_frames / total_time
        print(f"  Adjusted to match expected FPS: Scale factor = {scale_factor:.4f}, New FPS = {calculated_fps:.2f}")
    
    return time_array

In [ ]:
# Find all NPZ files in the preprocessed data directory
npz_files = list(PREPROCESSED_DATA_PATH.rglob('*.npz'))

print(f"Found {len(npz_files)} NPZ files in {PREPROCESSED_DATA_PATH}")

if npz_files:
    print("\nFirst 10 files:")
    for i, file_path in enumerate(npz_files[:10]):
        print(f"  {i+1}. {file_path.name}")
    
    # Select the first file for analysis
    selected_file = npz_files[0]
    print(f"\nSelecting first file for analysis: {selected_file}")
else:
    print("No NPZ files found. Creating a sample file for demonstration...")
    # Create a sample NPZ file for demonstration
    selected_file = Path('sample_demo.npz')
    
    # Create sample data matching the structure described
    num_frames = 450
    # Create realistic time deltas for 30 FPS (each delta = 1/30 second)
    time_deltas = np.ones(num_frames) * (1.0 / 30.0)
    sample_data = {
        'roi_full_face': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_forehead': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_left_eye': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_right_eye': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_nose': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_mouth': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_chin': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_right_cheek_50': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_left_cheek_280': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'roi_chin_199': np.random.rand(num_frames, 24, 24, 3).astype(np.float32),
        'ppg_values': np.random.randn(num_frames).astype(np.float32) * 10 + 100,
        'time_deltas': time_deltas.astype(np.float32),
        'landmarks': np.random.rand(num_frames, 468, 2).astype(np.float32),
        'meta_subject_id': np.array(1020),
        'meta_condition': np.array('after'),
        'meta_camera_type': np.array('FullHDwebcam'),
        'meta_view_type': np.array('front'),
        'meta_chunk_index': np.array(0),
        'meta_start_frame': np.array(0),
        'meta_end_frame': np.array(450),
        'meta_num_frames': np.array(450),
        'vital_upper_ap': np.array(113.0),
        'vital_lower_ap': np.array(78.0),
        'vital_saturation': np.array(98.0),
        'vital_hemoglobin': np.array(12.6),
        'vital_glycated_hemoglobin': np.array(5.56),
        'vital_cholesterol': np.array(4.3),
        'vital_respiratory': np.array(19.0),
        'vital_rigidity': np.array(10.79),
        'vital_pulse': np.array(100.0),
        'vital_stress': np.array(4.0),
    }
    
    np.savez(selected_file, **sample_data)
    print(f"Created sample file: {selected_file}")

In [ ]:
# Load the selected file
data = load_npz_file(selected_file)

if data is None:
    raise FileNotFoundError(f"Could not load {selected_file}")

# Analyze the structure
analyze_npz_structure(data)

# Extract metadata and ROI names
metadata = extract_metadata(data)
roi_names = extract_roi_names(data)

print("\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

print(f"\nROI Names ({len(roi_names)}):")
for roi in roi_names:
    print(f"  {roi}")

In [ ]:
# Extract PPG data
ppg_values = data['ppg_values']
time_deltas = data['time_deltas']

print(f"PPG Values: shape={ppg_values.shape}, dtype={ppg_values.dtype}")
print(f"Time Deltas: shape={time_deltas.shape}, dtype={time_deltas.dtype}")

# Compute time array with automatic scaling
print("\nComputing time array...")
time_array = compute_time_array(time_deltas, expected_fps=30)
total_time = time_array[-1] - time_array[0]

# Basic statistics
print(f"\nPPG Statistics:")
print(f"  Mean: {ppg_values.mean():.2f}")
print(f"  Std: {ppg_values.std():.2f}")
print(f"  Min: {ppg_values.min():.2f}")
print(f"  Max: {ppg_values.max():.2f}")
print(f"  Range: {ppg_values.max() - ppg_values.min():.2f}")

# Calculate time information
print(f"\nTime Information:")
print(f"  Total duration: {total_time:.2f} seconds")
print(f"  Frame rate: {len(ppg_values) / total_time:.2f} FPS")
print(f"  Expected FPS from metadata: {metadata.get('meta_num_frames', 'N/A') / (metadata.get('meta_end_frame', 1) - metadata.get('meta_start_frame', 0)) if metadata.get('meta_num_frames', 'N/A') != 'N/A' else 'N/A'}")

In [ ]:
# Plot PPG Signal
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'PPG Analysis - Subject {metadata.get("meta_subject_id", "N/A")} | {metadata.get("meta_condition", "N/A")} | {metadata.get("meta_camera_type", "N/A")}', fontsize=16)

# 1. Raw PPG Signal
ax1 = axes[0, 0]
ax1.plot(time_array, ppg_values, linewidth=1.5, color='red')
ax1.set_title('Raw PPG Signal', fontsize=14)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('PPG Value', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f9f9f9')

# 2. PPG Signal with Mean Line
ax2 = axes[0, 1]
ax2.plot(time_array, ppg_values, linewidth=1.5, color='red', alpha=0.7, label='PPG')
ax2.axhline(y=ppg_values.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {ppg_values.mean():.2f}')
ax2.set_title('PPG Signal with Mean', fontsize=14)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('PPG Value', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f9f9f9')

# 3. PPG Distribution
ax3 = axes[1, 0]
ax3.hist(ppg_values, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax3.set_title('PPG Value Distribution', fontsize=14)
ax3.set_xlabel('PPG Value', fontsize=12)
ax3.set_ylabel('Frequency', fontsize=12)
ax3.grid(True, alpha=0.3)
ax3.set_facecolor('#f9f9f9')

# 4. PPG Autocorrelation
ax4 = axes[1, 1]
corr = signal.correlate(ppg_values - ppg_values.mean(), ppg_values - ppg_values.mean(), mode='full')
corr = corr[len(corr)//2:]
lags = np.arange(len(corr)) * (time_array[1] - time_array[0]) if len(time_array) > 1 else np.arange(len(corr))
ax4.plot(lags, corr, linewidth=1.5, color='green')
ax4.set_title('PPG Autocorrelation', fontsize=14)
ax4.set_xlabel('Lag (s)', fontsize=12)
ax4.set_ylabel('Correlation', fontsize=12)
ax4.grid(True, alpha=0.3)
ax4.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

In [ ]:
# Detrend the PPG signal
detrended_ppg = detrend(ppg_values)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Original PPG
ax1 = axes[0]
ax1.plot(time_array, ppg_values, linewidth=1.5, color='red')
ax1.set_title('Original PPG Signal', fontsize=14)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('PPG Value', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f9f9f9')

# Detrended PPG
ax2 = axes[1]
ax2.plot(time_array, detrended_ppg, linewidth=1.5, color='blue')
ax2.set_title('Detrended PPG Signal', fontsize=14)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('Detrended PPG', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

In [ ]:
# Apply bandpass filter to extract heart rate
# Typical heart rate range: 40-240 BPM = 0.67-4 Hz
lowcut = 0.67  # 40 BPM
highcut = 4.0  # 240 BPM
fs = len(ppg_values) / total_time  # Sampling frequency
nyq = 0.5 * fs
low = lowcut / nyq
high = highcut / nyq

print(f"Filter parameters: fs={fs:.2f} Hz, nyq={nyq:.2f} Hz, low={low:.4f}, high={high:.4f}")

# Design bandpass filter
b, a = butter(3, [low, high], btype='band')
filtered_ppg = filtfilt(b, a, ppg_values)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Original vs Filtered
ax1 = axes[0]
ax1.plot(time_array, ppg_values, linewidth=1.5, color='red', alpha=0.7, label='Original')
ax1.plot(time_array, filtered_ppg, linewidth=1.5, color='green', label='Filtered')
ax1.set_title('PPG Signal: Original vs Bandpass Filtered', fontsize=14)
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('PPG Value', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_facecolor('#f9f9f9')

# Zoomed in view - use middle 20% of the data
ax2 = axes[1]
zoom_start_idx = int(len(time_array) * 0.4)
zoom_end_idx = int(len(time_array) * 0.6)
zoom_start = time_array[zoom_start_idx]
zoom_end = time_array[zoom_end_idx]
ax2.plot(time_array[zoom_start_idx:zoom_end_idx], ppg_values[zoom_start_idx:zoom_end_idx], linewidth=1.5, color='red', alpha=0.7, label='Original')
ax2.plot(time_array[zoom_start_idx:zoom_end_idx], filtered_ppg[zoom_start_idx:zoom_end_idx], linewidth=1.5, color='green', label='Filtered')
ax2.set_title(f'Zoomed View ({zoom_start:.2f}s - {zoom_end:.2f}s)', fontsize=14)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('PPG Value', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

In [ ]:
# Find peaks in filtered PPG
# Use minimum distance based on expected heart rate (60-180 BPM = 1-0.33 Hz)
min_peak_distance = fs * 0.33  # Minimum distance for 180 BPM
peaks, _ = find_peaks(filtered_ppg, height=np.percentile(filtered_ppg, 75), distance=min_peak_distance)

# Calculate heart rate
peak_times = time_array[peaks]
if len(peak_times) > 1:
    inter_peak_intervals = np.diff(peak_times)
    heart_rate = 60 / inter_peak_intervals  # Convert to BPM
    avg_heart_rate = heart_rate.mean()
    std_heart_rate = heart_rate.std()
    median_heart_rate = np.median(heart_rate)
else:
    avg_heart_rate = None
    std_heart_rate = None
    median_heart_rate = None

print(f"Peak Detection:")
print(f"  Number of peaks: {len(peaks)}")
print(f"  Minimum peak distance: {min_peak_distance:.1f} samples")
if avg_heart_rate is not None:
    print(f"  Average Heart Rate: {avg_heart_rate:.2f} BPM")
    print(f"  Median Heart Rate: {median_heart_rate:.2f} BPM")
    print(f"  Heart Rate Std: {std_heart_rate:.2f} BPM")
    print(f"  Heart Rate Range: {heart_rate.min():.2f} - {heart_rate.max():.2f} BPM")
    print(f"  Vital Pulse (from metadata): {metadata.get('vital_pulse', 'N/A')}")

# Plot with peaks
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(time_array, filtered_ppg, linewidth=1.5, color='green', label='Filtered PPG')
ax.plot(time_array[peaks], filtered_ppg[peaks], 'x', color='red', markersize=8, label='Peaks')
title = f'PPG Peaks Detection (HR: {avg_heart_rate:.1f} BPM avg, {median_heart_rate:.1f} BPM median)' if avg_heart_rate is not None else 'PPG Peaks Detection'
ax.set_title(title, fontsize=14)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Filtered PPG', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')
plt.show()

In [ ]:
if avg_heart_rate is not None:
    # Calculate instantaneous heart rate
    instant_hr_times = peak_times[1:]
    instant_hr = heart_rate
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Instantaneous HR
    ax1 = axes[0]
    ax1.plot(instant_hr_times, instant_hr, 'o-', color='purple', markersize=4, linewidth=1.5)
    ax1.axhline(y=avg_heart_rate, color='red', linestyle='--', linewidth=2, label=f'Average: {avg_heart_rate:.1f} BPM')
    ax1.axhline(y=median_heart_rate, color='orange', linestyle=':', linewidth=2, label=f'Median: {median_heart_rate:.1f} BPM')
    ax1.axhline(y=metadata.get('vital_pulse', avg_heart_rate), color='blue', linestyle=':', linewidth=2, label=f'Metadata Pulse: {metadata.get("vital_pulse", "N/A")}')
    ax1.set_title('Instantaneous Heart Rate', fontsize=14)
    ax1.set_xlabel('Time (s)', fontsize=12)
    ax1.set_ylabel('Heart Rate (BPM)', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_facecolor('#f9f9f9')
    
    # Histogram of HR
    ax2 = axes[1]
    ax2.hist(instant_hr, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
    ax2.axvline(x=avg_heart_rate, color='red', linestyle='--', linewidth=2, label=f'Average: {avg_heart_rate:.1f}')
    ax2.axvline(x=median_heart_rate, color='orange', linestyle=':', linewidth=2, label=f'Median: {median_heart_rate:.1f}')
    ax2.set_title('Heart Rate Distribution', fontsize=14)
    ax2.set_xlabel('Heart Rate (BPM)', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor('#f9f9f9')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Display information about ROI data
print("ROI Data Analysis:")
print("-" * 50)

for roi_name in roi_names[:5]:  # Show first 5 ROIs
    roi_data = data[roi_name]
    print(f"\n{roi_name}:")
    print(f"  Shape: {roi_data.shape}")
    print(f"  Frames: {roi_data.shape[0]}, Height: {roi_data.shape[1]}, Width: {roi_data.shape[2]}, Channels: {roi_data.shape[3]}")
    print(f"  Data type: {roi_data.dtype}")
    print(f"  Memory size: {roi_data.nbytes / 1024:.2f} KB")
    
    # Sample statistics for first frame
    first_frame = roi_data[0]
    print(f"  First frame mean: {first_frame.mean():.4f}")
    print(f"  First frame std: {first_frame.std():.4f}")

# Visualize a sample ROI frame
if roi_names:
    sample_roi = roi_names[0]  # First ROI
    sample_frame = data[sample_roi][0]  # First frame
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(f'Sample ROI Frame - {sample_roi} (Frame 0)', fontsize=14)
    
    for channel, color, title in zip(range(3), ['red', 'green', 'blue'], ['Red Channel', 'Green Channel', 'Blue Channel']):
        axes[channel].imshow(sample_frame[:, :, channel], cmap='gray')
        axes[channel].set_title(title, fontsize=12)
        axes[channel].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Extract landmarks
landmarks = data['landmarks']
print(f"\nLandmarks Data:")
print(f"  Shape: {landmarks.shape}")
print(f"  Frames: {landmarks.shape[0]}, Landmarks per frame: {landmarks.shape[1]}, Coordinates: {landmarks.shape[2]}")

# Visualize landmarks for first frame
first_frame_landmarks = landmarks[0]

fig, ax = plt.subplots(figsize=(8, 8))

# Plot landmarks
x_coords = first_frame_landmarks[:, 0]
y_coords = first_frame_landmarks[:, 1]

# Normalize coordinates for display
x_min, x_max = x_coords.min(), x_coords.max()
y_min, y_max = y_coords.min(), y_coords.max()

# Scale to image coordinates
scale = 100
x_scaled = ((x_coords - x_min) / (x_max - x_min)) * scale
y_scaled = ((y_coords - y_min) / (y_max - y_min)) * scale

ax.scatter(x_scaled, y_scaled, s=5, c='blue', alpha=0.6)
ax.set_title(f'Landmarks - Frame 0 ({len(x_coords)} points)', fontsize=14)
ax.set_xlabel('X (normalized)', fontsize=12)
ax.set_ylabel('Y (normalized)', fontsize=12)
ax.set_xlim(0, scale)
ax.set_ylim(scale, 0)  # Invert Y axis for image coordinates
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')
plt.show()

In [ ]:
print("=" * 60)
print("METADATA SUMMARY")
print("=" * 60)

# Subject information
print("\nSubject Information:")
print(f"  Subject ID: {metadata.get('meta_subject_id', 'N/A')}")
print(f"  Condition: {metadata.get('meta_condition', 'N/A')}")

# Camera information
print("\nCamera Information:")
print(f"  Camera Type: {metadata.get('meta_camera_type', 'N/A')}")
print(f"  View Type: {metadata.get('meta_view_type', 'N/A')}")

# Chunk information
print("\nChunk Information:")
print(f"  Chunk Index: {metadata.get('meta_chunk_index', 'N/A')}")
print(f"  Start Frame: {metadata.get('meta_start_frame', 'N/A')}")
print(f"  End Frame: {metadata.get('meta_end_frame', 'N/A')}")
print(f"  Number of Frames: {metadata.get('meta_num_frames', 'N/A')}")

# Vital signs
print("\nVital Signs:")
vital_keys = [k for k in metadata.keys() if k.startswith('vital_')]
for key in vital_keys:
    value = metadata[key]
    # Format the key name
    display_name = key.replace('vital_', '').replace('_', ' ').title()
    print(f"  {display_name}: {value}")

print("=" * 60)

In [ ]:
# Calculate mean intensity for each ROI over time
roi_means = {}
for roi_name in roi_names:
    roi_data = data[roi_name]
    # Calculate mean across spatial dimensions (H, W, C)
    roi_means[roi_name] = roi_data.mean(axis=(1, 2, 3))

# Plot mean intensity over time for all ROIs
fig, ax = plt.subplots(figsize=(14, 8))

colors = plt.cm.tab20(np.linspace(0, 1, len(roi_names)))

for i, roi_name in enumerate(roi_names):
    ax.plot(time_array, roi_means[roi_name], 
            linewidth=1.5, color=colors[i], 
            label=roi_name.replace('roi_', '').replace('_', ' ').title())

ax.set_title('Mean ROI Intensity Over Time', fontsize=16)
ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel('Mean Intensity', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate correlation between PPG and ROI mean intensities
correlations = {}
for roi_name in roi_names:
    corr = np.corrcoef(ppg_values, roi_means[roi_name])[0, 1]
    correlations[roi_name] = corr

# Sort by correlation strength
sorted_correlations = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("Correlation between PPG and ROI Mean Intensity:")
print("-" * 60)
for roi_name, corr in sorted_correlations:
    display_name = roi_name.replace('roi_', '').replace('_', ' ').title()
    print(f"  {display_name:25s}: {corr:.4f}")

# Plot correlations
fig, ax = plt.subplots(figsize=(12, 6))
roi_names_sorted = [item[0] for item in sorted_correlations]
corr_values = [item[1] for item in sorted_correlations]

colors = ['red' if c < 0 else 'green' for c in corr_values]
ax.bar(range(len(corr_values)), corr_values, color=colors, alpha=0.7)

# Custom x-axis labels
x_labels = [name.replace('roi_', '').replace('_', ' ').title() for name in roi_names_sorted]
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)

ax.set_title('Correlation: PPG vs ROI Mean Intensity', fontsize=14)
ax.set_ylabel('Correlation Coefficient', fontsize=12)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.grid(True, alpha=0.3, axis='y')
ax.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

# PPG statistics
print("\nPPG Signal:")
print(f"  Duration: {total_time:.2f} seconds")
print(f"  Frames: {len(ppg_values)}")
print(f"  Frame Rate: {len(ppg_values) / total_time:.2f} FPS")
print(f"  Mean: {ppg_values.mean():.2f}")
print(f"  Std: {ppg_values.std():.2f}")
print(f"  Min: {ppg_values.min():.2f}")
print(f"  Max: {ppg_values.max():.2f}")

if avg_heart_rate is not None:
    print(f"  Estimated Heart Rate: {avg_heart_rate:.2f} +- {std_heart_rate:.2f} BPM")
    print(f"  Median Heart Rate: {median_heart_rate:.2f} BPM")

# ROI statistics
print("\nROI Statistics:")
for roi_name in roi_names:
    roi_data = data[roi_name]
    mean_val = roi_data.mean()
    std_val = roi_data.std()
    display_name = roi_name.replace('roi_', '').replace('_', ' ').title()
    print(f"  {display_name:25s}: mean={mean_val:.2f}, std={std_val:.2f}")

# Landmarks statistics
print(f"\nLandmarks:")
print(f"  Total landmarks: {landmarks.shape[1]}")
print(f"  Frames: {landmarks.shape[0]}")

print("=" * 60)

In [ ]:
# Create a summary dictionary
analysis_summary = {
    'filename': str(selected_file.name),
    'subject_id': metadata.get('meta_subject_id', 'N/A'),
    'condition': metadata.get('meta_condition', 'N/A'),
    'camera_type': metadata.get('meta_camera_type', 'N/A'),
    'view_type': metadata.get('meta_view_type', 'N/A'),
    'num_frames': len(ppg_values),
    'duration_seconds': float(total_time),
    'frame_rate': float(len(ppg_values) / total_time),
    'ppg_mean': float(ppg_values.mean()),
    'ppg_std': float(ppg_values.std()),
    'ppg_min': float(ppg_values.min()),
    'ppg_max': float(ppg_values.max()),
    'estimated_heart_rate': float(avg_heart_rate) if avg_heart_rate is not None else None,
    'heart_rate_std': float(std_heart_rate) if std_heart_rate is not None else None,
    'median_heart_rate': float(median_heart_rate) if median_heart_rate is not None else None,
    'metadata_pulse': float(metadata.get('vital_pulse', 'N/A')),
    'num_rois': len(roi_names),
    'roi_names': roi_names,
    'correlations': {k: float(v) for k, v in correlations.items()},
    'timestamp': str(np.datetime64('now')),
}

# Save summary to JSON
summary_file = selected_file.with_suffix('.analysis.json')
with open(summary_file, 'w') as f:
    json.dump(analysis_summary, f, indent=2)

print(f"\nAnalysis summary saved to: {summary_file}")

In [ ]:
print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print(f"File analyzed: {selected_file}")
print(f"Summary saved to: {summary_file}")
print("\nTo analyze another file, change the 'selected_file' variable and re-run the cells.")